# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
# Carga directa desde la tabla subida al Workspace
bronce = spark.table("workspace.default.ventas_export_ventas_export")
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

Filas en el archivo: 59
Columnas: ['venta_id', 'fecha', 'tienda_id', 'producto_id', 'cantidad', 'monto', 'medio_pago', 'canal']


## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO


In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# 1. Transformación inicial y numeración de filas en vista intermedia
spark.sql("""
CREATE OR REPLACE TEMP VIEW bronce_limpia AS
SELECT 
  venta_id,
  fecha,
  CAST(tienda_id AS INT) AS tienda_id,
  producto_id,
  cantidad,
  CASE 
    WHEN monto LIKE '%.%,%' THEN CAST(REPLACE(REPLACE(monto, '.', ''), ',', '.') AS DOUBLE)
    WHEN monto LIKE '%,%' THEN CAST(REPLACE(monto, ',', '.') AS DOUBLE)
    ELSE TRY_CAST(monto AS DOUBLE)
  END AS monto,
  medio_pago,
  canal,
  ROW_NUMBER() OVER (PARTITION BY venta_id ORDER BY fecha ASC) as rn
FROM bronce_ventas
""")

# 2. Creación de la capa Oro (oro_fact_ventas)
spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS
SELECT 
  venta_id,
  fecha,
  tienda_id,
  producto_id,
  cantidad,
  monto,
  medio_pago
FROM bronce_limpia
WHERE rn = 1
  AND tienda_id IN (1, 2, 3)
  AND monto IS NOT NULL
""")

# 3. Creación de la capa Cuarentena (cuarentena_ventas)
spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS
SELECT 
  venta_id,
  fecha,
  tienda_id,
  producto_id,
  cantidad,
  monto,
  medio_pago,
  canal,
  CASE 
    WHEN tienda_id NOT IN (1, 2, 3) OR tienda_id IS NULL THEN 'Tienda no permitida'
    WHEN monto IS NULL THEN 'Monto invalido o faltante'
    WHEN rn > 1 THEN 'Venta duplicada'
  END AS motivo
FROM bronce_limpia
WHERE rn > 1 
   OR tienda_id NOT IN (1, 2, 3) 
   OR tienda_id IS NULL 
   OR monto IS NULL
""")

DataFrame[]

## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO


## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
Para identificar los registros repetidos sin perder información, usamos la función de ventana ROW_NUMBER() OVER (PARTITION BY venta_id ORDER BY fecha ASC). La primera entrada válida (rn = 1) se envió a la tabla de hecho oro_fact_ventas. Todas las repeticiones posteriores (rn > 1) se filtraron y se enviaron directamente a cuarentena_ventas bajo el motivo 'Venta duplicada', asegurando que el hecho no contenga duplicados pero manteniendo la auditoría completa de lo que llegó.

2. ¿Qué hicieron con un monto que venía en texto?
Aplicamos una lógica de limpieza con REPLACE y TRY_CAST a DOUBLE. Si el monto venía en formato de texto pero con números (como "12,50" o "1.250,00"), se corrigieron los puntos y comas para convertirlo a un valor numérico real. Si venía con texto no convertible como "N/D" o "pendiente", la conversión devolvió NULL, lo que activó la regla para mandar esa fila a cuarentena_ventas etiquetada como 'Monto invalido o faltante'.

3. ¿Por qué una tienda se quedó fuera?
Las reglas del negocio establecen que solo son válidas las transacciones de las tiendas 1 (Centro), 2 (Escalón) y 3 (Soyapango). Cualquier registro con un tienda_id distinto (como 4, 5, 6, 77, 88, 99) o con valores nulos se excluyó de la capa Oro y se registró en cuarentena_ventas con el motivo 'Tienda no permitida', protegiendo el tablero de Finanzas de datos fuera de alcance.

4. ¿Dónde quedó la columna que no estaba en el contrato?
La columna canal no formaba parte del contrato de datos acordado con Finanzas para la tabla oro_fact_ventas. Por esa razón, se omitió de la proyección SELECT al construir la capa Oro. Sin embargo, la columna no se eliminó de la fuente original ni de la capa Bronce, y se conservó dentro de cuarentena_ventas para no perder la trazabilidad de los datos descartados.

5. Si corren el notebook otra vez, ¿el hecho crece?
No, el hecho se mantiene idéntico. Todo el flujo está diseñado con vistas temporales mediante CREATE OR REPLACE TEMP VIEW. Esto garantiza idempotencia: no importa cuántas veces se ejecute el cuaderno, las vistas se reconstruyen desde cero a partir de la fuente Bronce y siempre generarán exactamente las mismas 27 filas en Oro y 32 en Cuarentena sin duplicar ni acumular registros.

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio